# Deep Learning Cheat Sheet
## Every Key Pattern from the 3-Week Course — One Notebook

Use this as your go-to reference after completing the course.  
Each section is self-contained — copy-paste what you need.

---

## 1. The Single Neuron (Pure Python)

```
output = activation(weights · inputs + bias)
```

This is the atom of deep learning. Everything else is layers of this.

In [1]:
import numpy as np

# === SINGLE NEURON ===
def neuron(x, w, b, activation='relu'):
    z = np.dot(x, w) + b
    if activation == 'relu':
        return np.maximum(0, z)
    elif activation == 'sigmoid':
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))
    return z  # linear

# Example
x = np.array([1.0, 0.5])
w = np.random.randn(2) * 0.1
b = 0.0
print(f"ReLU output:    {neuron(x, w, b, 'relu'):.4f}")
print(f"Sigmoid output: {neuron(x, w, b, 'sigmoid'):.4f}")

ReLU output:    0.0000
Sigmoid output: 0.4977


## 2. Activation Functions — When to Use What

| Activation | Use When | Watch Out For |
|-----------|----------|---------------|
| **ReLU** | Default for hidden layers | Dead neurons (output always 0) |
| **Sigmoid** | Binary output (last layer) | Vanishing gradients in deep nets |
| **Tanh** | Centered output needed | Same vanishing gradient problem |
| **Linear** | Regression output | No non-linearity added |

**Rule of thumb:** ReLU for hidden layers, sigmoid for binary classification output, linear for regression output.

In [2]:
# === ACTIVATION FUNCTIONS AND THEIR DERIVATIVES ===

def relu(z):       return np.maximum(0, z)
def relu_d(z):     return (z > 0).astype(float)

def sigmoid(z):    
    s = 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))
    return s
def sigmoid_d(z):  
    s = sigmoid(z)
    return s * (1 - s)

# KEY INSIGHT: Sigmoid max gradient = 0.25 (vanishes when chained)
#              ReLU gradient = 1 when active (never vanishes!)
z_test = np.array([0.0])
print(f"Sigmoid gradient at z=0: {sigmoid_d(z_test)[0]:.2f} (max possible)")
print(f"ReLU gradient at z=0.1: {relu_d(np.array([0.1]))[0]:.0f} (always 1 when active)")

Sigmoid gradient at z=0: 0.25 (max possible)
ReLU gradient at z=0.1: 1 (always 1 when active)


## 3. Backpropagation — The Chain Rule Pattern

```
Forward:  x → z=Wx+b → a=act(z) → loss
Backward: dL/dW = dL/da · da/dz · dz/dW
```

Every backward pass follows this pattern. It's dynamic programming for gradients.

In [3]:
# === BACKPROP IN 10 LINES ===
# Given: layer with weights W, bias b, activation

def forward(x, W, b):
    z = x @ W + b
    a = relu(z)
    return z, a

def backward(x, z, da, W):
    dz = da * relu_d(z)           # gradient through activation
    dW = (x.T @ dz) / len(x)     # gradient for weights
    db = np.mean(dz, axis=0)      # gradient for bias
    dx = dz @ W.T                 # gradient for input (pass to prev layer)
    return dW, db, dx

# Update rule: W -= learning_rate * dW
print("Backprop is just the chain rule applied layer by layer. That's it.")

Backprop is just the chain rule applied layer by layer. That's it.


## 4. Initialization — He vs Xavier

| Init Method | Formula | Use With |
|------------|---------|----------|
| **He** | `std = sqrt(2/fan_in)` | ReLU activations |
| **Xavier** | `std = sqrt(1/fan_in)` | Sigmoid/Tanh activations |
| **Zeros** | Never for weights! | Only for biases |

Wrong init = network won't train. This is the #1 silent killer.

In [4]:
# === INITIALIZATION PATTERNS ===

# Pure Python
def he_init(fan_in, fan_out):
    return np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / fan_in)

def xavier_init(fan_in, fan_out):
    return np.random.randn(fan_in, fan_out) * np.sqrt(1.0 / fan_in)

# PyTorch equivalents
import torch
import torch.nn as nn

layer = nn.Linear(64, 32)
nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')  # He
# nn.init.xavier_normal_(layer.weight)  # Xavier

print(f"He init std for fan_in=64: {np.sqrt(2.0/64):.4f}")
print(f"Xavier std for fan_in=64:  {np.sqrt(1.0/64):.4f}")

He init std for fan_in=64: 0.1768
Xavier std for fan_in=64:  0.1250


## 5. PyTorch Training Loop — The Template

This is the pattern you'll use for every PyTorch project. Memorize it.

In [5]:
# === THE UNIVERSAL PYTORCH TRAINING LOOP ===
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Define model
class Net(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

# 2. Create model, optimizer, loss
model = Net(2, 32, 1)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# 3. Training loop (MEMORIZE THIS PATTERN)
def train_epoch(model, X_train, y_train, optimizer, criterion):
    model.train()                        # Enable dropout/batchnorm training mode
    predictions = model(X_train)         # Forward pass
    loss = criterion(predictions, y_train)  # Compute loss
    optimizer.zero_grad()                # Clear old gradients
    loss.backward()                      # Compute new gradients (autograd!)
    optimizer.step()                     # Update weights
    return loss.item()

# 4. Validation (NO gradients!)
def validate(model, X_val, y_val, criterion):
    model.eval()                         # Disable dropout/batchnorm
    with torch.no_grad():                # No gradient computation
        predictions = model(X_val)
        loss = criterion(predictions, y_val)
        accuracy = ((predictions > 0.5).float() == y_val).float().mean()
    return loss.item(), accuracy.item()

print("Template ready. Modify input_dim, hidden_dim, output_dim for your task.")

Template ready. Modify input_dim, hidden_dim, output_dim for your task.


## 6. Debugging Checklist — When Your Model Won't Train

**In order of likelihood:**

1. **Learning rate too high/low?** → Try 0.001 (Adam) or 0.01 (SGD) first
2. **Data normalized?** → `X = (X - X.mean()) / X.std()`
3. **Correct loss function?** → MSE for regression, BCE for classification
4. **Can it overfit 1 batch?** → If not, architecture or code bug
5. **Gradients flowing?** → Check `param.grad` is not None or all zeros
6. **Labels correct?** → Plot X vs y, sanity check
7. **BatchNorm before activation** → `Linear → BN → ReLU`, never `Linear → ReLU → BN`

In [6]:
# === QUICK DEBUGGING TOOLS ===

def check_gradients(model):
    "Check if gradients are flowing properly."
    print("Gradient check:")
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_mean = param.grad.abs().mean().item()
            grad_max = param.grad.abs().max().item()
            status = "OK" if grad_mean > 1e-7 else "DEAD"
            print(f"  {name:30s} mean={grad_mean:.2e} max={grad_max:.2e} [{status}]")
        else:
            print(f"  {name:30s} NO GRADIENT!")

def check_overfit_one_batch(model, X_batch, y_batch, epochs=200):
    "If model cannot overfit 1 batch, something is wrong."
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    for i in range(epochs):
        model.train()
        loss = criterion(model(X_batch), y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Final loss after {epochs} epochs on 1 batch: {loss.item():.6f}")
    if loss.item() > 0.1:
        print("WARNING: Cannot overfit one batch! Check architecture/code.")
    else:
        print("OK: Model can overfit. Training issue is elsewhere.")

print("Use check_gradients(model) after one backward pass.")
print("Use check_overfit_one_batch(model, X[:32], y[:32]) to sanity check.")

Use check_gradients(model) after one backward pass.
Use check_overfit_one_batch(model, X[:32], y[:32]) to sanity check.


## 7. VAE — The Generative Model Pattern

```
Encoder: x → μ, log(σ²)
Reparameterize: z = μ + σ · ε,  ε ~ N(0,1)
Decoder: z → x_reconstructed
Loss: reconstruction_error + KL_divergence
```

In [7]:
# === VAE TEMPLATE (PyTorch) ===

class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim), nn.Sigmoid(),
        )
    
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)       # Sample from N(0,1)
        return mu + std * eps              # The trick!
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    recon = nn.functional.mse_loss(x_hat, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl

# Generate new samples: z = torch.randn(n, latent_dim) → decoder(z)
print("VAE template ready. Adjust dims for your data.")

VAE template ready. Adjust dims for your data.


## 8. Deployment — Gradio in 10 Lines

```python
import gradio as gr

def predict(input_data):
    # Your model inference here
    return model(input_data)

demo = gr.Interface(
    fn=predict,
    inputs=gr.Slider(-3, 3),
    outputs=gr.Plot()
)
demo.launch()  # Opens a web app!
```

**Deploy to Hugging Face Spaces:**
1. Create repo at huggingface.co/new-space
2. Upload: `app.py`, `model.pth`, `requirements.txt`
3. Done — you have a URL to share

## 9. The 7 Curriculum Truths — Quick Reference

| # | Truth | One-Liner |
|---|-------|-----------|
| 1 | Universal Approximation | One wide layer CAN fit anything; depth is more EFFICIENT |
| 2 | ReLU Changed Everything | Gradient = 0 or 1, never shrinks → deep nets can train |
| 3 | Overparameterization Helps | More params than data? GD + structure = implicit regularization |
| 4 | NNs Don't Understand | They minimize loss. They find statistics, not meaning |
| 5 | Backprop = Chain Rule | Dynamic programming for gradients. Nothing mystical |
| 6 | Geometry Matters | Flat minima generalize. Sharp minima don't |
| 7 | Capacity != Performance | Init + normalization + data quality beat raw size |

## 10. What to Learn Next

| Topic | Why | Best Starting Resource |
|-------|-----|----------------------|
| **CNNs** | Image recognition at scale | fast.ai Practical Deep Learning, Lessons 1-3 |
| **Transformers** | Foundation of modern LLMs | Andrej Karpathy "Let's build GPT" (YouTube) |
| **Diffusion Models** | State-of-art image generation | Lilian Weng blog: "What are Diffusion Models?" |
| **Fine-tuning LLMs** | Practical GenAI applications | Hugging Face PEFT documentation |
| **MLOps** | Production deployment | Made With ML (madewithml.com) |

---
*Keep this notebook open while you work. Every pattern you need is here.*